# Chapitre 1 — Télécharger les données (clé Kaggle)

**🎯 Objectif :** récupérer le jeu de données RSNA depuis Kaggle, soit en **échantillon**
(quelques patients, pour suivre tout le cours rapidement), soit en **complet**. C'est le
prérequis des chapitres 2.5, 3, 4 et 5.

**⏱ Durée :** échantillon ~1–3 min ; dataset complet plusieurs **heures** (~314 Go).

**Avant de lancer :**
1. Avoir accepté les règles de la compétition sur <https://www.kaggle.com/competitions/rsna-breast-cancer-detection/rules> (sinon erreur 403).
2. Token Kaggle : sur kaggle.com → *Settings* → *Create New Token*, puis dépose le fichier **`access_token`** (préfixe `KGAT_`) dans le dossier **`.kaggle/`** à la racine du dépôt. C'est le format récent, le seul qui fonctionne avec `kaggle` 2.x (l'ancien `kaggle.json` username+clé renvoie une 401 avec ces tokens). Le dossier est monté en lecture seule dans le conteneur.

**Deux sections indépendantes** ci-dessous :
- **Section A — Échantillon** (recommandé, par défaut) → `data/in/rsna_sample`
- **Section B — Dataset complet** (long, désactivé par défaut) → `data/in/rsna`

Les cellules communes (auth + labels) se lancent d'abord, puis tu choisis la section à exécuter.

In [1]:
import os
from course_utils import data_in
# Vérifie qu'un identifiant Kaggle est présent AVANT d'importer kaggle (l'import authentifie aussitôt).
os.environ.setdefault('KAGGLE_CONFIG_DIR', os.path.expanduser('~/.kaggle'))
KDIR = os.environ['KAGGLE_CONFIG_DIR']
have_token = os.path.isfile(os.path.join(KDIR, 'access_token'))   # token récent KGAT_ (recommandé)
have_json  = os.path.isfile(os.path.join(KDIR, 'kaggle.json'))    # ancien format username + key
have_env   = bool(os.environ.get('KAGGLE_API_TOKEN') or
                  (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')))
assert have_token or have_json or have_env, (
    "Identifiant Kaggle absent. Dépose ton token dans .kaggle/access_token (format récent "
    "KGAT_, recommandé) ou un kaggle.json (ancien format). Voir le README.")
import kaggle  # déclenche l'authentification
print('Authentification Kaggle OK',
      '(access_token)' if have_token else '(kaggle.json)' if have_json else '(env)')

Authentification Kaggle OK (access_token)


In [ ]:
import glob, zipfile

COMP = 'rsna-breast-cancer-detection'
DATA_DIR = data_in('rsna')   # <repo>/data/in/rsna -> volume persistant
os.makedirs(DATA_DIR, exist_ok=True)

def _unzip_all(folder):
    """Décompresse puis supprime tous les .zip d'un dossier.

    L'API Kaggle enveloppe TOUJOURS ses téléchargements dans un .zip, même pour un
    seul fichier (ex. `train.csv.zip`, `1234.dcm.zip`) -> on décompresse après CHAQUE
    appel à competition_download_file(s), puis on supprime le zip (sinon on double le
    stockage pour rien, une fois le .zip et une fois le fichier extrait).
    """
    for z in glob.glob(os.path.join(folder, '*.zip')):
        print('décompression', os.path.basename(z))
        with zipfile.ZipFile(z) as zf:
            zf.extractall(folder)
        os.remove(z)

print('Cible :', DATA_DIR)

In [3]:
# --- Labels seuls (léger) : de quoi explorer avant de tout télécharger ---
import pandas as pd
train_csv = os.path.join(DATA_DIR, 'train.csv')
if not os.path.exists(train_csv):
    kaggle.api.competition_download_file(COMP, 'train.csv', path=DATA_DIR, quiet=False)
    _unzip_all(DATA_DIR)
df = pd.read_csv(train_csv)
print('train.csv :', df.shape)
print('prévalence cancer :', round(df['cancer'].mean(), 4))
df.head()

100%|████████████████████████████████████████████████████████████████| 556k/556k [00:00<00:00, 1.22MB/s]


décompression train.csv.zip
train.csv : (54706, 14)
prévalence cancer : 0.0212


,site_id,patient_id,image_id,laterality,view,age,cancer,biopsy,invasive,BIRADS,implant,density,machine_id,difficult_negative_case
0,2,10006,462822612,L,CC,61.0,0,0,0,NaN,0,NaN,29,False
1,2,10006,1459541791,L,MLO,61.0,0,0,0,NaN,0,NaN,29,False
2,2,10006,1864590858,R,MLO,61.0,0,0,0,NaN,0,NaN,29,False
3,2,10006,1874946579,R,CC,61.0,0,0,0,NaN,0,NaN,29,False
4,2,10011,220375232,L,CC,55.0,0,0,0,0.0,0,NaN,21,True


## Section A — Échantillon stratifié (recommandé)

Sous-ensemble de quelques patients (~50–80 Mo), idéal pour parcourir tout le cours
rapidement. On reprend la logique de `extract_download.py` : une ligne par patient
(label = `max` du cancer), un `train_test_split` **stratifié** sur la prévalence, puis
la garantie d'au moins un patient avec cancer et un sain. Les fichiers sont récupérés
un par un via `competition_download_file`. Résultat dans `data/in/rsna_sample`.

> C'est la section à lancer pour la suite du cours. La cellule est **idempotente**
> (relançable) et affiche le temps d'extraction.

In [ ]:
# Section A — échantillon stratifié -> data/in/rsna_sample
import time
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

N_PATIENTS = 4                          # nombre de patients de l'échantillon
SAMPLE_DIR = data_in('rsna_sample')     # <repo>/data/in/rsna_sample
os.makedirs(os.path.join(SAMPLE_DIR, 'train_images'), exist_ok=True)

t0 = time.time()
# Patients ayant les 4 vues (examen complet) -> exploitables au ch2.5.
NEED = {('L', 'CC'), ('L', 'MLO'), ('R', 'CC'), ('R', 'MLO')}
views_by_pid = df.groupby('patient_id').apply(lambda g: set(zip(g['laterality'], g['view'])))
complete = views_by_pid[views_by_pid.apply(NEED.issubset)].index
grouped = df[df['patient_id'].isin(complete)].groupby('patient_id')['cancer'].max().reset_index()

# Échantillon stratifié sur la prévalence du cancer (déterministe).
selected, _ = train_test_split(
    grouped, train_size=N_PATIENTS, stratify=grouped['cancer'], random_state=42)
for label in (1, 0):                       # garantir >=1 cancer et >=1 sain
    if not (selected['cancer'] == label).any():
        extra = grouped[grouped['cancer'] == label].sample(1, random_state=42)
        selected = pd.concat([selected, extra], ignore_index=True)

pids = sorted(int(p) for p in selected['patient_id'])
print('Patients (stratifiés) :', pids,
      '| cancer:', int(selected['cancer'].sum()),
      '| sains:', int((selected['cancer'] == 0).sum()))

rows = df[df['patient_id'].isin(pids)]
rows.to_csv(os.path.join(SAMPLE_DIR, 'train.csv'), index=False)
# Barre de progression sur le téléchargement image par image.
for _, r in tqdm(rows.iterrows(), total=len(rows), desc='Téléchargement DICOM'):
    pid, iid = int(r['patient_id']), int(r['image_id'])
    dst = os.path.join(SAMPLE_DIR, 'train_images', str(pid))
    os.makedirs(dst, exist_ok=True)
    if os.path.exists(os.path.join(dst, f'{iid}.dcm')):
        continue                           # reprise : skip si déjà présent
    kaggle.api.competition_download_file(
        COMP, f'train_images/{pid}/{iid}.dcm', path=dst, quiet=True)
    _unzip_all(dst)

n = len(glob.glob(os.path.join(SAMPLE_DIR, 'train_images', '**', '*.dcm'), recursive=True))
print(f'{n} DICOM dans {SAMPLE_DIR}')
print(f'⏱ Extraction (échantillon) terminée en {time.time() - t0:.1f} s '
      f'— prêt pour le ch2.5 (INPUT_DIR = data/in/rsna_sample).')

In [ ]:
# Aperçu : on lit un des DICOM téléchargés et on l'affiche tel quel (avant tout prétraitement).
import pydicom, numpy as np, matplotlib.pyplot as plt

_dcms = sorted(glob.glob(os.path.join(SAMPLE_DIR, 'train_images', '**', '*.dcm'), recursive=True))
assert _dcms, "Aucun DICOM — exécute d'abord la cellule de téléchargement ci-dessus."
_ds = pydicom.dcmread(_dcms[0])
_img = _ds.pixel_array.astype('float32')
if _ds.PhotometricInterpretation == 'MONOCHROME1':     # certains scanners stockent l'image inversée
    _img = _img.max() - _img

plt.figure(figsize=(4, 6))
plt.imshow(_img, cmap='gray'); plt.axis('off')
plt.title(f"DICOM brut — {os.path.relpath(_dcms[0], SAMPLE_DIR)}\n{_img.shape[1]}x{_img.shape[0]} ({_ds.PhotometricInterpretation})")
plt.show()
print(f"{len(_dcms)} DICOM téléchargés. Image brute : {_img.shape}, "
      f"intensités {_img.min():.0f}–{_img.max():.0f}.")
print("-> au chapitre 2.5 elle sera convertie, recadrée et redimensionnée pour GMIC.")

## Section B — Dataset complet (optionnel, long)

**Tout** le dataset DICOM (~314 Go, plusieurs heures, pic disque ~628 Go), téléchargé
via `competition_download_files`. Résultat dans `data/in/rsna`.

> ⚠️ Désactivé par défaut (`RUN_FULL = False`) pour qu'un « Run All » ne déclenche pas
> 314 Go par accident. Mets `RUN_FULL = True` **et exécute uniquement la cellule
> ci-dessous** quand tu veux vraiment le dataset complet.

In [ ]:
# Section B — dataset RSNA complet -> data/in/rsna  (~314 Go, plusieurs heures)
RUN_FULL = False     # <-- passe à True pour lancer réellement le téléchargement complet

if not RUN_FULL:
    print("Section B désactivée (RUN_FULL = False). Mets RUN_FULL = True pour télécharger "
          "tout le dataset (~314 Go). Pour le cours, l'échantillon de la Section A suffit.")
else:
    import time
    t0 = time.time()
    if not os.path.isdir(os.path.join(DATA_DIR, 'train_images')):
        print('Téléchargement du dataset COMPLET (~314 Go)… (long)')
        kaggle.api.competition_download_files(COMP, path=DATA_DIR, quiet=False)
        _unzip_all(DATA_DIR)
    n = len(glob.glob(os.path.join(DATA_DIR, 'train_images', '**', '*.dcm'), recursive=True))
    print(f'Dataset complet dans {DATA_DIR} ({n} DICOM)')
    print(f'⏱ Extraction (complet) terminée en {time.time() - t0:.1f} s '
          f'— prêt pour le ch2.5 (INPUT_DIR = data/in/rsna).')